### Load Silver data

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

silver_orders = spark.table("workspace.ecommerce_project.silver_orders")

### Business Aggregation

### 1. Category Performance

In [0]:
category_performance = (
    silver_orders
    .groupBy("category")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("order_amount").alias("gross_order_value"),
        F.sum("final_amount").alias("total_revenue"),
        F.avg("final_amount").alias("average_order_value")
    )
    # beacuse spark provides in E7 in the numeric that 10^7
    .withColumn(
        "gross_order_value",
        F.col("gross_order_value").cast(DecimalType(18,2))
    )
    .withColumn(
        "total_revenue",
        F.col("total_revenue").cast(DecimalType(18,2))
    )
    .withColumn(
        "average_order_value",
        F.col("average_order_value").cast(DecimalType(18,2))
    )
    .orderBy(F.desc("total_revenue"))
)

In [0]:
category_performance.show()

+-----------+------------+--------------+-----------------+-------------+-------------------+
|   category|total_orders|total_quantity|gross_order_value|total_revenue|average_order_value|
+-----------+------------+--------------+-----------------+-------------+-------------------+
|ELECTRONICS|         293|          1649|      32520082.89|  25737392.10|           87840.93|
|       HOME|         104|           627|       4039869.11|   3077370.72|           29590.10|
|    FASHION|         101|           583|       1780921.85|   1283584.46|           12708.76|
|     SPORTS|          47|           251|        825402.48|    697817.40|           14847.18|
|      BOOKS|          63|           349|        445564.35|    347831.09|            5521.13|
|    GROCERY|          58|           310|        314321.20|    243922.71|            4205.56|
|     BEAUTY|          59|           284|        193582.53|    145304.30|            2462.78|
+-----------+------------+--------------+-----------------+-

In [0]:
category_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.ecommerce_project.gold_category_performance"
    )

### 2. Order Status Summary

In [0]:
order_status_summary = (
    silver_orders
    .groupBy("order_status")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("final_amount").alias("total_revenue"),
        F.avg("final_amount").alias("average_order_value")
    )
    .withColumn(
        "total_revenue",
        F.col("total_revenue").cast(DecimalType(18,2))
    )
    .withColumn(
        "average_order_value",
        F.col("average_order_value").cast(DecimalType(18,2))
    )
    .orderBy(F.desc("total_orders"))
)

In [0]:
order_status_summary.show()

+------------+------------+-------------+-------------------+
|order_status|total_orders|total_revenue|average_order_value|
+------------+------------+-------------+-------------------+
|   Delivered|         419|  18703000.51|           44637.23|
|   Cancelled|          76|   2797161.65|           36804.76|
|     Shipped|          71|   3033636.82|           42727.28|
|    Returned|          66|   2087053.70|           31622.03|
|   Confirmed|          62|   2385504.89|           38475.89|
|      Placed|          31|   2526865.21|           81511.78|
+------------+------------+-------------+-------------------+



In [0]:
order_status_summary.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.ecommerce_project.gold_order_status_summary"
)

### 3. Payment Performance

In [0]:
payment_performance = (
    silver_orders
    .groupBy(
        "payment_method",
        "payment_status"
        )
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("final_amount").alias("total_amount"),
        F.avg("final_amount").alias("average_order_value")
    )
    .withColumn(
        "total_amount",
        F.col("total_amount").cast(DecimalType(18,2))
    )
    .withColumn(
        "average_order_value",
        F.col("average_order_value").cast(DecimalType(18,2))
    )
    .orderBy("payment_method",F.desc("total_amount"))
)

In [0]:
payment_performance.show()

+----------------+--------------+------------+------------+-------------------+
|  payment_method|payment_status|total_orders|total_amount|average_order_value|
+----------------+--------------+------------+------------+-------------------+
|CASH ON DELIVERY|          PAID|          74|  2530014.71|           34189.39|
|CASH ON DELIVERY|       PENDING|          28|   928768.97|           33170.32|
|CASH ON DELIVERY|      REFUNDED|          23|   620799.02|           26991.26|
|     CREDIT CARD|          PAID|          86|  4060048.76|           47209.87|
|     CREDIT CARD|      REFUNDED|          27|  1529104.96|           56633.52|
|     CREDIT CARD|       PENDING|           7|   138761.27|           19823.04|
|     CREDIT CARD|        FAILED|           1|    10800.38|           10800.38|
|      DEBIT CARD|          PAID|          94|  3277704.25|           34869.19|
|      DEBIT CARD|       PENDING|           7|   956465.88|          136637.98|
|      DEBIT CARD|      REFUNDED|       

In [0]:
payment_performance.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.ecommerce_project.gold_payment_performance")

### 4. Regional Sales

In [0]:
regional_sales = (
    silver_orders
    .groupBy("shipping_state")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("final_amount").alias("total_revenue"),
        F.avg("Final_amount").alias("average_order_value")
    )
    .withColumn("total_revenue", 
                F.col("total_revenue").cast(DecimalType(18,2))
    )
    .withColumn("average_order_value", 
                F.col("average_order_value").cast(DecimalType(18,2))
                )
    .orderBy(F.desc("total_revenue"))
)

In [0]:
regional_sales.show()

+--------------+------------+--------------+-------------+-------------------+
|shipping_state|total_orders|total_quantity|total_revenue|average_order_value|
+--------------+------------+--------------+-------------+-------------------+
|    Tamil Nadu|         161|           925|   6767397.78|           42033.53|
|   Maharashtra|         100|           544|   5707263.27|           57072.63|
|     Telangana|          67|           395|   3399705.12|           50741.87|
|     Rajasthan|          62|           357|   3035472.21|           48959.23|
|       Gujarat|          65|           370|   3009820.78|           46304.94|
|     Karnataka|          61|           331|   2587097.08|           42411.43|
|        Kerala|          49|           260|   1967379.75|           40150.61|
| Uttar Pradesh|          67|           394|   1895295.10|           28287.99|
|   West Bengal|          44|           238|   1734879.54|           39429.08|
|         Delhi|          49|           239|   14289

In [0]:
regional_sales.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce_project.gold_regional_sales")

In [0]:
city_sales = (
    silver_orders
    .groupBy(
        "shipping_state",
        "shipping_city"
    )
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("final_amount").alias("total_revenue")
    )
    .withColumn(
        "total_revenue",
        F.col("total_revenue").cast(DecimalType(18,2))
    )
    .orderBy("shipping_state")
)

In [0]:
city_sales.show()

+--------------+-------------+------------+-------------+
|shipping_state|shipping_city|total_orders|total_revenue|
+--------------+-------------+------------+-------------+
|         Delhi|        Delhi|          49|   1428912.15|
|       Gujarat|    Ahmedabad|          65|   3009820.78|
|     Karnataka|    Bengaluru|          61|   2587097.08|
|        Kerala|        Kochi|          49|   1967379.75|
|   Maharashtra|       Mumbai|          53|   2826109.35|
|   Maharashtra|         Pune|          47|   2881153.92|
|     Rajasthan|       Jaipur|          62|   3035472.21|
|    Tamil Nadu|   Coimbatore|          59|   1346994.41|
|    Tamil Nadu|      Madurai|          58|   3534423.81|
|    Tamil Nadu|      Chennai|          44|   1885979.56|
|     Telangana|    Hyderabad|          67|   3399705.12|
| Uttar Pradesh|      Lucknow|          67|   1895295.10|
|   West Bengal|      Kolkata|          44|   1734879.54|
+--------------+-------------+------------+-------------+



In [0]:
silver_orders.groupBy("shipping_city").count().show()

+-------------+-----+
|shipping_city|count|
+-------------+-----+
|    Bengaluru|   61|
|      Lucknow|   67|
|    Hyderabad|   67|
|         Pune|   47|
|      Madurai|   58|
|       Jaipur|   62|
|      Kolkata|   44|
|        Delhi|   49|
|   Coimbatore|   59|
|      Chennai|   44|
|        Kochi|   49|
|    Ahmedabad|   65|
|       Mumbai|   53|
+-------------+-----+



### 5. Daily Sales

In [0]:
daily_sales = (
    silver_orders
    .groupBy(
        "order_date"
    )
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("final_amount").alias("total_revenue"),
        F.avg("final_amount").alias("average_order_value")
    )
    .orderBy("order_date")
)

In [0]:
daily_sales.show()

+----------+------------+--------------+------------------+-------------------+
|order_date|total_orders|total_quantity|     total_revenue|average_order_value|
+----------+------------+--------------+------------------+-------------------+
|2025-01-01|           1|             2|          14889.45|           14889.45|
|2025-01-02|           2|             8|           5481.17|           2740.585|
|2025-01-03|           1|             7|            8954.4|             8954.4|
|2025-01-04|           2|            13|         964668.73|         482334.365|
|2025-01-05|           2|            15|           3685.75|           1842.875|
|2025-01-07|           1|            10|          71100.45|           71100.45|
|2025-01-08|           1|            10|            711.84|             711.84|
|2025-01-09|           3|            12|          97964.61|           32654.87|
|2025-01-10|           1|             9|           6231.89|            6231.89|
|2025-01-12|           3|            11|

In [0]:
daily_sales.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce_project.gold_daily_sales")

### KPI

### KPI 1: Total Orders

In [0]:
total_orders = silver_orders.count()

### KPI 2: Total Revenue

In [0]:
total_revenue = (
    silver_orders
    .agg(
        F.sum("final_amount").alias("total_revenue")
    )
    .withColumn("total_revenue",F.col("total_revenue").cast(DecimalType(18,2)))
    .first()["total_revenue"]
)

In [0]:
print(total_revenue)
# 3.1533222780000005E7

31533222.78


### KPI 3: Total Quantity Sold

In [0]:
total_quantity = (
    silver_orders
    .agg(
        F.sum("quantity").alias("total_quantity")
    )
    .first()["total_quantity"]
)

In [0]:
total_quantity

4053

### KPI 4: Average Order Value

In [0]:
average_order_value = (
    silver_orders
    .agg(
        F.avg("final_amount").alias("average_order_value")
    )
    .first()["average_order_value"]
)

### KPI 5: Delivered Order Rate

In [0]:
delivered_orders = (
    silver_orders
    .filter(
        F.col("order_status") == "Delivered"
    )
    .count()
)

delivered_order_rate = (
    delivered_orders / total_orders
) * 100

In [0]:
delivered_order_rate

57.793103448275865

### KPI 6: Cancellation Rate

In [0]:
cancelled_orders = (
    silver_orders
    .filter(
        F.col("order_status") == "Cancelled"
    )
    .count()
)

cancellation_rate = (
    cancelled_orders / total_orders
) * 100

### KPI 7: Return Rate

In [0]:
returned_orders = (
    silver_orders
    .filter(
        F.col("order_status") == "Returned"
    )
    .count()
)

return_rate = (
    returned_orders / total_orders
) * 100

### KPI 8: Payment Success Rate

In [0]:
successful_payments = (
    silver_orders
    .filter(
        F.col("payment_status") == "PAID"
    )
    .count()
)
payment_success_rate = (
    successful_payments / total_orders
) * 100

### Combining all the cell into the single one

In [0]:
kpi_metrics = (
    silver_orders
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("final_amount").alias("total_revenue"),
        F.sum("quantity").alias("total_quantity"),
        F.avg("final_amount").alias("average_order_value"),
        F.sum(
        F.when(
            F.col("order_status") == "Delivered",1
        ).otherwise(0).alias("")
        ).alias("delivered_orders"),

        F.sum(
        F.when(
            F.col("order_status") == "Cancelled",1
        ).otherwise(0)
        ).alias("cancelled_orders"),

        F.sum(
        F.when(
            F.col("order_status") == "Returned",1
        ).otherwise(0)
        ).alias("returned_orders"),

        F.sum(
        F.when(
            F.col("payment_status") == "PAID",1
        ).otherwise(0)
        ).alias("successful_payments")
    )
)

In [0]:
kpi_metrics.show()

+------------+--------------------+--------------+-------------------+----------------+----------------+---------------+-------------------+
|total_orders|       total_revenue|total_quantity|average_order_value|delivered_orders|cancelled_orders|returned_orders|successful_payments|
+------------+--------------------+--------------+-------------------+----------------+----------------+---------------+-------------------+
|         725|3.1533222780000005E7|          4053| 43494.100386206905|             419|              76|             66|                527|
+------------+--------------------+--------------+-------------------+----------------+----------------+---------------+-------------------+



In [0]:
kpi_metrics = (
    kpi_metrics
    .withColumn(
        "total_revenue",
        F.col("total_revenue").cast(DecimalType(18,2))
    )
    .withColumn(
        "average_order_value",
        F.col("average_order_value").cast(DecimalType(18,2))
    )
    .withColumn(
        "delivered_rate",
        F.round(
            (F.col("delivered_orders")/F.col("total_orders"))*100,
            2
        )
    )
    .withColumn(
        "cancellation_rate",
        F.round(
            (F.col("cancelled_orders")/F.col('total_orders'))*100,
            2
        )
    )
    .withColumn(
        "return_rate",
            F.round(
            (F.col("returned_orders")/F.col("total_orders"))*100,
            2
        )
    )
    .withColumn(
        "payment_sucessful_rate",
        F.round(
            (F.col("successful_payments")/F.col("total_orders"))*100,
            2
        )
    )
)

In [0]:
kpi_metrics.show()

+------------+-------------+--------------+-------------------+----------------+----------------+---------------+-------------------+--------------+-----------------+-----------+----------------------+
|total_orders|total_revenue|total_quantity|average_order_value|delivered_orders|cancelled_orders|returned_orders|successful_payments|delivered_rate|cancellation_rate|return_rate|payment_sucessful_rate|
+------------+-------------+--------------+-------------------+----------------+----------------+---------------+-------------------+--------------+-----------------+-----------+----------------------+
|         725|  31533222.78|          4053|           43494.10|             419|              76|             66|                527|         57.79|            10.48|        9.1|                 72.69|
+------------+-------------+--------------+-------------------+----------------+----------------+---------------+-------------------+--------------+-----------------+-----------+--------------

In [0]:
kpi_metrics.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.ecommerce_project.gold_kpis"
)

### Reporting tables

### 1. Monthly Sales Report

In [0]:
monthly_sales_report = (
    silver_orders.withColumn(
        "month_start",
        F.date_trunc(
            "month",
            F.col("order_date")
        )
    )
)

In [0]:
monthly_sales_report.select("order_date","month_start").orderBy("month_start").show(monthly_sales_report.count())

+----------+-------------------+
|order_date|        month_start|
+----------+-------------------+
|2025-01-05|2025-01-01 00:00:00|
|2025-01-31|2025-01-01 00:00:00|
|2025-01-31|2025-01-01 00:00:00|
|2025-01-27|2025-01-01 00:00:00|
|2025-01-09|2025-01-01 00:00:00|
|2025-01-29|2025-01-01 00:00:00|
|2025-01-20|2025-01-01 00:00:00|
|2025-01-12|2025-01-01 00:00:00|
|2025-01-28|2025-01-01 00:00:00|
|2025-01-17|2025-01-01 00:00:00|
|2025-01-02|2025-01-01 00:00:00|
|2025-01-22|2025-01-01 00:00:00|
|2025-01-24|2025-01-01 00:00:00|
|2025-01-10|2025-01-01 00:00:00|
|2025-01-04|2025-01-01 00:00:00|
|2025-01-26|2025-01-01 00:00:00|
|2025-01-19|2025-01-01 00:00:00|
|2025-01-13|2025-01-01 00:00:00|
|2025-01-12|2025-01-01 00:00:00|
|2025-01-29|2025-01-01 00:00:00|
|2025-01-23|2025-01-01 00:00:00|
|2025-01-17|2025-01-01 00:00:00|
|2025-01-07|2025-01-01 00:00:00|
|2025-01-08|2025-01-01 00:00:00|
|2025-01-21|2025-01-01 00:00:00|
|2025-01-22|2025-01-01 00:00:00|
|2025-01-09|2025-01-01 00:00:00|
|2025-01-2

In [0]:
monthly_sales_report = (
    monthly_sales_report
    .groupBy(
        "month_start",
        "category"
    )
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("order_amount").alias("gross_order_value"),
        F.sum("final_amount").alias("total_amount"),
        F.avg("final_amount").alias("average_order_value")

    )
)

In [0]:
monthly_sales_report = (
    monthly_sales_report
    .withColumn(
        "gross_order_value",
        F.col("gross_order_value").cast(DecimalType(18,2))
    )
    .withColumn(
        "total_amount",
        F.col("total_amount").cast(DecimalType(18,2))
    )
    .withColumn(
        "average_order_value",
        F.col("average_order_value").cast(DecimalType(18,2))
    )
)

In [0]:
monthly_sales_report.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.ecommerce_project.gold_monthly_sales_report"
    )

### 2. Regional Sales Report

In [0]:
regional_sales_report = (
    silver_orders
    .groupBy(
        "shipping_state",
        "category"
    )
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("final_amount").alias("total_revenue"),
        F.avg("final_amount").alias("average_order_value")
    )
)

In [0]:
regional_sales_report = (
    regional_sales_report
    .withColumn(
        "total_revenue",
        F.col("total_revenue")
        .cast(DecimalType(18, 2))
    )
    .withColumn(
        "average_order_value",
        F.col("average_order_value")
        .cast(DecimalType(18, 2))
    )
)

In [0]:
regional_sales_report.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.ecommerce_project.gold_regional_sales_report"
    )

### 3. Order Fulfillment Report

In [0]:
order_fulfillment_report = (
    silver_orders
    .withColumn(
        "month_start",
        F.date_trunc(
            "month",
            F.col("order_date")
        )
    )
    .groupBy(
        "month_start",
        "order_status"
    )
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("final_amount").alias("total_revenue")

    )
)

In [0]:
order_fulfillment_report = (
    order_fulfillment_report
    .withColumn(
        "total_revenue",
        F.col("total_revenue")
        .cast(DecimalType(18, 2))
    )
)

In [0]:
order_fulfillment_report.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.ecommerce_project.gold_order_fulfillment_report"
    )

### Product Performance Report

In [0]:
product_performance_report = (
    silver_orders
    .groupBy(
        "product_id",
        "product_name",
        "category"
    )
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("final_amount").alias("total_revenue"),
        F.avg("final_amount").alias("average_order_value")
    )
)

In [0]:
product_performance_report = (
    product_performance_report
    .withColumn(
        "total_revenue",
        F.col("total_revenue")
        .cast(DecimalType(18, 2))
    )
    .withColumn(
        "average_order_value",
        F.col("average_order_value")
        .cast(DecimalType(18, 2))
    )
)

In [0]:
product_performance_report.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.ecommerce_project.gold_product_performance_report"
    )

###  Dashboard Tables

### 1. Dashboard KPI Table

In [0]:
gold_kpis = spark.table(
    "workspace.ecommerce_project.gold_kpis"
)

In [0]:
gold_kpis.show()

+------------+-------------+--------------+-------------------+----------------+----------------+---------------+-------------------+--------------+-----------------+-----------+----------------------+
|total_orders|total_revenue|total_quantity|average_order_value|delivered_orders|cancelled_orders|returned_orders|successful_payments|delivered_rate|cancellation_rate|return_rate|payment_sucessful_rate|
+------------+-------------+--------------+-------------------+----------------+----------------+---------------+-------------------+--------------+-----------------+-----------+----------------------+
|         725|  31533222.78|          4053|           43494.10|             419|              76|             66|                527|         57.79|            10.48|        9.1|                 72.69|
+------------+-------------+--------------+-------------------+----------------+----------------+---------------+-------------------+--------------+-----------------+-----------+--------------

In [0]:
(
    gold_kpis.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.ecommerce_project.gold_dashborad_kpis"
    )
)

### 2. Monthly Revenue Dashboard Table

In [0]:
monthly_sales_report = spark.table(
    "workspace.ecommerce_project.gold_monthly_sales_report"
)

In [0]:
monthly_sales_report.show()

+-------------------+-----------+------------+--------------+-----------------+------------+-------------------+
|        month_start|   category|total_orders|total_quantity|gross_order_value|total_amount|average_order_value|
+-------------------+-----------+------------+--------------+-----------------+------------+-------------------+
|2025-10-01 00:00:00|     SPORTS|           5|            28|        119205.27|   116428.60|           23285.72|
|2025-04-01 00:00:00|       HOME|          12|            74|        427835.31|   342774.83|           28564.57|
|2025-02-01 00:00:00|    FASHION|           8|            46|        163975.34|    93236.34|           11654.54|
|2025-08-01 00:00:00|       HOME|          11|            71|        559896.56|   472088.21|           42917.11|
|2025-07-01 00:00:00|    FASHION|          11|            70|        163121.74|   136964.26|           12451.30|
|2025-04-01 00:00:00|    FASHION|           6|            33|        100807.72|    63558.12|    

In [0]:
dashboard_monthly_revenue = (
    monthly_sales_report
    .groupBy(
        "month_start"
    )
    .agg(
        F.sum("total_amount").alias("total_revenue")
    )
    .orderBy("month_start")
)

In [0]:
dashboard_monthly_revenue.show()

+-------------------+-------------+
|        month_start|total_revenue|
+-------------------+-------------+
|2025-01-01 00:00:00|   3772019.18|
|2025-02-01 00:00:00|   1957747.05|
|2025-03-01 00:00:00|   2437614.42|
|2025-04-01 00:00:00|   2138183.81|
|2025-05-01 00:00:00|   1434813.60|
|2025-06-01 00:00:00|   3138255.49|
|2025-07-01 00:00:00|   5434028.67|
|2025-08-01 00:00:00|   1953838.81|
|2025-09-01 00:00:00|   1579780.04|
|2025-10-01 00:00:00|   2159572.92|
|2025-11-01 00:00:00|   1227253.25|
|2025-12-01 00:00:00|   4300115.54|
+-------------------+-------------+



In [0]:
dashboard_monthly_revenue.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce_project.gold_dashboard_monthly_revenue")

### 3. Category Revenue Dashboard Table

In [0]:
category_performance.show()

+-----------+------------+--------------+-----------------+-------------+-------------------+
|   category|total_orders|total_quantity|gross_order_value|total_revenue|average_order_value|
+-----------+------------+--------------+-----------------+-------------+-------------------+
|ELECTRONICS|         293|          1649|      32520082.89|  25737392.10|           87840.93|
|       HOME|         104|           627|       4039869.11|   3077370.72|           29590.10|
|    FASHION|         101|           583|       1780921.85|   1283584.46|           12708.76|
|     SPORTS|          47|           251|        825402.48|    697817.40|           14847.18|
|      BOOKS|          63|           349|        445564.35|    347831.09|            5521.13|
|    GROCERY|          58|           310|        314321.20|    243922.71|            4205.56|
|     BEAUTY|          59|           284|        193582.53|    145304.30|            2462.78|
+-----------+------------+--------------+-----------------+-

In [0]:
dashboard_category_revenue = (
    category_performance
    .select(
        "category",
        "total_revenue"
    )
    .orderBy(F.desc("total_revenue"))
)

In [0]:
dashboard_category_revenue.show()

+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|ELECTRONICS|  25737392.10|
|       HOME|   3077370.72|
|    FASHION|   1283584.46|
|     SPORTS|    697817.40|
|      BOOKS|    347831.09|
|    GROCERY|    243922.71|
|     BEAUTY|    145304.30|
+-----------+-------------+



In [0]:
dashboard_category_revenue.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce_project.gold_dashboard_category_revenue")

### 4. Regional Sales Dashboard Table

In [0]:
regional_sales_report.show()

+--------------+-----------+------------+--------------+-------------+-------------------+
|shipping_state|   category|total_orders|total_quantity|total_revenue|average_order_value|
+--------------+-----------+------------+--------------+-------------+-------------------+
|     Karnataka|     SPORTS|           6|            30|    108587.69|           18097.95|
| Uttar Pradesh|ELECTRONICS|          27|           153|   1417292.16|           52492.30|
|     Telangana|      BOOKS|           9|            68|     79478.10|            8830.90|
|   Maharashtra|       HOME|          16|           100|    444859.80|           27803.74|
|    Tamil Nadu|    FASHION|          23|           143|    271369.36|           11798.67|
|     Rajasthan|ELECTRONICS|          26|           130|   2445625.25|           94062.51|
|   West Bengal|       HOME|           5|            31|    240271.97|           48054.39|
|     Karnataka|     BEAUTY|           7|            26|     15672.77|            2238.97|

In [0]:
regional_sales_report = (
    spark.table("workspace.ecommerce_project.gold_regional_sales_report")
)

In [0]:
dashboard_regional_sales = (
    regional_sales_report
    .groupBy(
        "shipping_state"
    )
    .agg(
        F.sum("total_orders").alias("total_orders"),
        F.sum("total_quantity").alias("total_quantity"),
        F.sum("total_revenue").alias("total_revenue")
    )
)

In [0]:
dashboard_regional_sales.show()

+--------------+------------+--------------+-------------+
|shipping_state|total_orders|total_quantity|total_revenue|
+--------------+------------+--------------+-------------+
|     Telangana|          67|           395|   3399705.12|
|         Delhi|          49|           239|   1428912.15|
|    Tamil Nadu|         161|           925|   6767397.78|
|       Gujarat|          65|           370|   3009820.78|
|        Kerala|          49|           260|   1967379.75|
|     Rajasthan|          62|           357|   3035472.21|
|   Maharashtra|         100|           544|   5707263.27|
|     Karnataka|          61|           331|   2587097.08|
|   West Bengal|          44|           238|   1734879.54|
| Uttar Pradesh|          67|           394|   1895295.10|
+--------------+------------+--------------+-------------+



In [0]:
dashboard_regional_sales.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce_project.gold_dashboard_regional_sales")

### 5. Order Status Dashboard Table

In [0]:
order_fulfillment_report.show()

+-------------------+------------+------------+--------------+-------------+
|        month_start|order_status|total_orders|total_quantity|total_revenue|
+-------------------+------------+------------+--------------+-------------+
|2025-10-01 00:00:00|     Shipped|          10|            63|    256835.52|
|2025-12-01 00:00:00|   Cancelled|           9|            56|    455827.34|
|2025-11-01 00:00:00|   Cancelled|           5|            27|    116147.22|
|2025-09-01 00:00:00|   Delivered|          37|           220|    896228.09|
|2025-12-01 00:00:00|   Delivered|          39|           221|   3126439.42|
|2025-01-01 00:00:00|   Delivered|          35|           184|   1459964.89|
|2025-11-01 00:00:00|   Delivered|          28|           136|    537671.03|
|2025-07-01 00:00:00|     Shipped|           5|            39|   1241566.22|
|2025-05-01 00:00:00|    Returned|           7|            36|    354566.56|
|2025-09-01 00:00:00|   Cancelled|           6|            42|    481117.57|

In [0]:
order_fulfillment_report = spark.table(
    "workspace.ecommerce_project.gold_order_fulfillment_report"
)

In [0]:
dashboard_order_status = (
    order_fulfillment_report
    .groupBy("order_status")
    .agg(
        F.sum("total_orders").alias("total_orders")
    )
    .orderBy(F.desc("total_orders"))
)

In [0]:
dashboard_order_status.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce_project.gold_dashboard_order_status")

### 6. Top Products Dashboard Table

In [0]:
product_performance_report.show()

+----------+-------------------+-----------+------------+--------------+-------------+-------------------+
|product_id|       product_name|   category|total_orders|total_quantity|total_revenue|average_order_value|
+----------+-------------------+-----------+------------+--------------+-------------+-------------------+
|  PROD1064|        Cricket Bat|     SPORTS|           9|            42|    136451.78|           15161.31|
|  PROD1073|              Mouse|ELECTRONICS|          10|            45|     67234.25|            6723.43|
|  PROD1066|              Books|      BOOKS|          11|            59|     88718.46|            8065.31|
|  PROD1036|       Water Bottle|       HOME|          12|            74|     25702.14|            2141.85|
|  PROD1075|       Cotton Shirt|    FASHION|           6|            39|     58571.67|            9761.95|
|  PROD1048|  Bluetooth Speaker|ELECTRONICS|           9|            64|    420639.71|           46737.75|
|  PROD1100|      Mixer Grinder|     

In [0]:
product_performance_report = spark.table(
    "workspace.ecommerce_project.gold_product_performance_report"
)

In [0]:
dashboard_top_products = (
    product_performance_report
    .select(
        "product_id",
        "product_name",
        "category",
        "total_quantity",
        "total_revenue"
    )
    .orderBy(F.desc("total_revenue"))
    .limit(10)
)

In [0]:
dashboard_top_products.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce_project.gold_dashboard_top_products")
